# Data Cleaning

This notebook details data cleaning steps applied to the initial raw FSA data pull (as completed in `./01_data_exploration.ipynb`, saved in `../data/raw/fsa_london_establishments.json`).  

### Load raw data & flatten into DataFrame

The data is loaded from the saved JSON file - raw data pull save issues should surface here, so this also acts as a simple verification step.  


In [1]:
import json, pandas as pd

# load JSON and store in variable
with open("../data/raw/fsa_london_establishments.json") as f:
    all_establishments = json.load(f)

# check record count
print(f"Loaded {len(all_establishments)} records.")

# store in DataFrame
df = pd.json_normalize(all_establishments) # pd.json_normalize rather than pd.DataFrame
    # required due to nested fields (e.g. geocode -> long/lat, scores -> individual score values)
    # non-normalised DF would keep these nested fields as dict objects inside single cells
    # ...which is not especially helpful

# inspect stored DF
df.info()

Loaded 81217 records.
<class 'pandas.DataFrame'>
RangeIndex: 81217 entries, 0 to 81216
Data columns (total 28 columns):
 #   Column                         Non-Null Count  Dtype  
---  ------                         --------------  -----  
 0   AddressLine1                   81217 non-null  str    
 1   AddressLine2                   81217 non-null  str    
 2   AddressLine3                   81217 non-null  str    
 3   AddressLine4                   81217 non-null  str    
 4   BusinessName                   81217 non-null  str    
 5   BusinessType                   81217 non-null  str    
 6   BusinessTypeID                 81217 non-null  int64  
 7   ChangesByServerID              81217 non-null  int64  
 8   Distance                       0 non-null      object 
 9   FHRSID                         81217 non-null  int64  
 10  LocalAuthorityBusinessID       81217 non-null  str    
 11  LocalAuthorityCode             81217 non-null  str    
 12  LocalAuthorityEmailAddress     8121

-> 81,217 rows loaded matches the most recent raw pull: this is good news.   

Nested values have populated into the DataFrame as expected, e.g. `geocode.longitude`.  

In terms of null values found by column:
- `Distance` is a 'dead' column, completely empty because no search radius was provided in the API request.  
- `scores.` columns have approx. 70k non-null values, roughly aligning with the number of premises with a 'gradable' rating.  
- `geocode.longitude` and `geocode.latitude` have approx. 67k non-null values. The missing values are potentially mobile businesses - this will need some further investigation.  
- All other key columns are fully populated.  

### Investigate null values:

The null values have some likely potential explanations, as above, however in order to double-check the partially filled `scores.` and `geocode.` columns:

In [2]:
# null nested 'scores' values check:

# get the null percentage of a group of values
def get_missing_percentage(group_data):
    return group_data.isna().mean()

# apply to nested Hygiene scores, grouping by RatingKey
df.groupby("RatingKey")["scores.Hygiene"].apply(get_missing_percentage)

RatingKey
fhrs_0_en-gb                      0.013333
fhrs_1_en-gb                      0.003580
fhrs_2_en-gb                      0.007011
fhrs_3_en-gb                      0.010556
fhrs_4_en-gb                      0.014971
fhrs_5_en-gb                      0.010927
fhrs_awaitinginspection_en-gb     1.000000
fhrs_awaitingpublication_en-gb    1.000000
fhrs_exempt_en-gb                 1.000000
Name: scores.Hygiene, dtype: float64

-> The awaiting/exempt ratings are fully empty - as expected.  

Numeric, 'gradable' ratings having approx. 1% of values missing was not expected - a small but non-zero number.  

In terms of the borough spread of missing nested score values:

In [3]:
# filter to numeric rating, missing hygiene score:
missing_scores = df[df["RatingValue"].isin(["0","1","2","3","4","5"]) & df["scores.Hygiene"].isna()]

# output total and borough counts
print(len(missing_scores)) 
print(missing_scores["LocalAuthorityName"].value_counts())

805
LocalAuthorityName
Westminster                   172
Croydon                        82
Sutton                         57
Waltham Forest                 57
Barnet                         55
City of London Corporation     48
Kingston-Upon-Thames           48
Lambeth                        46
Enfield                        45
Wandsworth                     44
Hackney                        35
Southwark                      30
Hammersmith and Fulham         28
Richmond-Upon-Thames           20
Merton                         13
Haringey                        9
Kensington and Chelsea          9
Bexley                          3
Camden                          3
Hillingdon                      1
Name: count, dtype: int64


-> 805 total found

Spread is across 20 boroughs, rather than an input/administrative gap in a single borough.

In terms of the top-10 by missing percentage:

In [4]:
# isolate gradable (numeric) ratings
gradable_df = df[df["RatingValue"].isin(["0","1","2","3","4","5"])]

# return Series for missing count (from above step) and total gradable count
missing_by_authority = missing_scores["LocalAuthorityName"].value_counts()
total_by_authority = gradable_df["LocalAuthorityName"].value_counts()

# Series / Series (aligns by LA Name index) to get pct, DESC order
missing_rate = (missing_by_authority / total_by_authority).sort_values(ascending=False)

# print top 10
print(missing_rate.head(10))

LocalAuthorityName
Sutton                        0.048635
Kingston-Upon-Thames          0.037915
Westminster                   0.032644
Waltham Forest                0.030695
Croydon                       0.029088
City of London Corporation    0.027923
Barnet                        0.023246
Enfield                       0.021226
Lambeth                       0.019159
Wandsworth                    0.018197
Name: count, dtype: float64


-> 805 gradable establishments (approx. 1.1%) are missing sub-scores. Missing rate varies by borough, with Sutton (4.9%) being the highest.  

This could be inconsistent Local Authority reporting, or could be driven by another factor.  

I'll also check `BusinessType`, in case there is a pattern:

In [5]:
# extract missing/gradable counts:
missing_by_type = missing_scores["BusinessType"].value_counts()
total_by_type = gradable_df["BusinessType"].value_counts()

# assemble DF, aligning Series by LA Name label
business_type_summary = pd.DataFrame({
    "missing": missing_by_type,
    "total": total_by_type
})

# fillna(0) avoids NaN if a business category has no missing scores
business_type_summary["missing"] = business_type_summary["missing"].fillna(0) 

# add 'rate' column with percentage missing
business_type_summary["rate"] = business_type_summary["missing"] / business_type_summary["total"]

print(business_type_summary.sort_values("rate", ascending=False))

                                       missing  total      rate
BusinessType                                                   
Takeaway/sandwich shop                   173.0   8526  0.020291
Restaurant/Cafe/Canteen                  431.0  25175  0.017120
Hotel/bed & breakfast/guest house         13.0    943  0.013786
Pub/bar/nightclub                         36.0   3707  0.009711
Retailers - supermarkets/hypermarkets     15.0   2259  0.006640
Caring Premises                           27.0   4281  0.006307
School/college/university                 17.0   3226  0.005270
Other catering premises                   26.0   6019  0.004320
Retailers - other                         58.0  13519  0.004290
Mobile caterer                             7.0   1927  0.003633
Manufacturers/packers                      2.0    846  0.002364
Importers/Exporters                        0.0    123  0.000000
Distributors/Transporters                  0.0    428  0.000000
Farmers/growers                         

-> Takeaways/Sandwich Shops have the highest proportion at 2.0%, with Restaurant/Cafe/Canteen having the highest total (431).  

No dramatic outliers are seen from this approach, where business type clearly explains the missing values. It may be the case that some takeaway / restaurant type premises have simpler inspections where the establishments are particularly simple operations, or it may be something like a re-inspection focuses only on the final rating.    

At this point - given the relatively low totals and prevalence - I'm not too concerned by these missing values, despite the lack of a clear explanation. Where the nested `scores` values are needed in a model, these rows can be dropped.  

Next, I am checking the c. 14k missing `geocode` values, using the same 3 groupings (Local Authority, Rating Key, Business Type):

In [6]:
# isolate rows with missing longitude value
missing_geo = df[df["geocode.longitude"].isna()]
print(len(missing_geo))

14069


In [7]:
# 1. by Local Authority, rate with actual counts:

# get value counts, grouped by LA, for full and missing data DFs
geo_missing_la = missing_geo["LocalAuthorityName"].value_counts()
geo_total_la = df["LocalAuthorityName"].value_counts()

# build dataframe, handle LAs with N/As, calculate missing %
geo_summary = pd.DataFrame({
    "missing": geo_missing_la, 
    "total": geo_total_la
})
geo_summary["missing"] = geo_summary["missing"].fillna(0)
geo_summary["rate"] = geo_summary["missing"] / geo_summary["total"]

# output
print(geo_summary.sort_values("rate", ascending=False))

                            missing  total      rate
LocalAuthorityName                                  
Richmond-Upon-Thames            498   1578  0.315589
Bromley                         738   2475  0.298182
Merton                          437   1566  0.279055
Lewisham                        622   2482  0.250604
Barnet                          706   2859  0.246939
Harrow                          464   1903  0.243826
Barking and Dagenham            353   1459  0.241947
Croydon                         746   3158  0.236225
Havering                        476   2021  0.235527
Greenwich                       546   2391  0.228356
Bexley                          402   1770  0.227119
Ealing                          799   3733  0.214037
Redbridge                       460   2158  0.213160
Hounslow                        483   2337  0.206675
Wandsworth                      586   2897  0.202278
Islington                       509   2591  0.196449
Hackney                         461   2448  0.

In [8]:
# 2. by Ratings Key, rate only

geo_rating_key_missing_pct = df.groupby("RatingKey")["geocode.longitude"].apply(lambda x: x.isna().mean())

print(geo_rating_key_missing_pct.sort_values(ascending=False))

RatingKey
fhrs_awaitinginspection_en-gb     0.434997
fhrs_awaitingpublication_en-gb    0.250000
fhrs_5_en-gb                      0.171206
fhrs_exempt_en-gb                 0.160589
fhrs_4_en-gb                      0.118968
fhrs_1_en-gb                      0.066826
fhrs_0_en-gb                      0.060000
fhrs_3_en-gb                      0.059987
fhrs_2_en-gb                      0.043065
Name: geocode.longitude, dtype: float64


In [9]:
# 3. by Business Type, rate with actual counts:

geo_missing_bt = missing_geo["BusinessType"].value_counts()
geo_total_bt = df["BusinessType"].value_counts()

geo_summary_bt = pd.DataFrame({
    "missing": geo_missing_bt, 
    "total": geo_total_bt
})
geo_summary_bt["missing"] = geo_summary_bt["missing"].fillna(0)
geo_summary_bt["rate"] = geo_summary_bt["missing"] / geo_summary_bt["total"]

# output
print(geo_summary_bt.sort_values("rate", ascending=False))

                                       missing  total      rate
BusinessType                                                   
Mobile caterer                            1715   2591  0.661907
Other catering premises                   5130   7987  0.642294
Farmers/growers                             16     31  0.516129
Manufacturers/packers                      355   1125  0.315556
Importers/Exporters                         82    275  0.298182
Distributors/Transporters                  167    666  0.250751
School/college/university                  662   3487  0.189848
Caring Premises                            621   4660  0.133262
Retailers - other                         1657  16665  0.099430
Restaurant/Cafe/Canteen                   2565  27188  0.094343
Pub/bar/nightclub                          299   3860  0.077461
Takeaway/sandwich shop                     640   9392  0.068143
Hotel/bed & breakfast/guest house           63    969  0.065015
Retailers - supermarkets/hypermarkets   

-> 14.1k rows have missing `geocode` (latitude/longitude) values. From inspecting the 3 groupings:

One signal - from Business Type - is especially clear. Business without a clear, fixed location - such as mobile caterers or caterers more generally - have a much higher rate of missing geo values. Conversely, establishments with a clearly defined, fixed premises (i.e. restaurants, cafes, takeaways and supermarkets) have much lower rates of missing lat/long info.  

There is a lot a variance across Local Authority boroughs, however I would expect this to be a reflection of the first point: a different mix of business types drives different rates of missing geo values, e.g. where outer boroughs (such as Richmond-Upon-Thames in the #1 spot) may have more mobile businesses than the restaurant/pub/sandwich shop hospitality-focused central boroughs.  

In terms of ratings, 'awaiting' score values have significantly higher rates of missing values, presumably where a business is new and the location data has not been collected/input yet.  

Based on this, there is at least some logical explanation for many of the missing values, but this does not account for all of them. Rather than dropping and excluding these ~14k rows, I'll look to backfill geocode via the postcode lookup where possible.

I therefore need to look at `PostCode` values to check if they are useable as a backup/replacement for missing `geocode` values.

UK postcodes have two parts: an _outward_ code (area + district, usually a digit, but some central London districts add a letter, e.g. W1A, EC1A) and an _inward_ code (always one digit + two letters, e.g. 1AA). The regex used below accounts for the optional district letter accordingly:

In [10]:
# inspect postcodes values

import numpy as np

# get 'outward code' using regex - this is the first part of a postcode before a space
    # i.e. 1-2 letters, then a digit, then an optional extra letter/digit
    # the joys of London postcodes
outward = r"[A-Z]{1,2}\d[A-Z\d]?"

# use outward code to look for the three postcode patterns used
full_pattern    = rf"^{outward}\s*\d[A-Z]{{2}}$"   # e.g. SW1A 1AA, full postcode
sector_pattern = rf"^{outward}\s+\d$"              # e.g. W3 7, a postcode sector
outward_pattern = rf"^{outward}$"                  # e.g. SE18, just the outward code

# clean: all upper, strip leading/trailing space
cleaned = df["PostCode"].str.upper().str.strip()

# categorise with select: works as an if/else
df["postcode_tier"] = np.select(
    condlist=[
        cleaned.eq(""), # empty string
        cleaned.str.match(full_pattern, na=False), # full poscode
        cleaned.str.match(sector_pattern, na=False), # sector only
        cleaned.str.match(outward_pattern, na=False), # outward code only
    ],
    choicelist=["blank", "full", "sector", "outward"],
    default="unclassified", # something else
)

# print the value counts by postcode type:
print(df["postcode_tier"].value_counts())


postcode_tier
full            70488
outward          8504
sector           1113
blank            1094
unclassified       18
Name: count, dtype: int64


-> from this check, the approximate counts are:

- 70.4k full postcodes - these are the most useful
- 9.6k partial postcodes (either sector or outward code only) - may still be useful
- 1.1k blank postcodes - missing

And 18 unclassified, as below:

In [11]:
# print unclassified postcodes
print(df.loc[df["postcode_tier"] == "unclassified", "PostCode"])

3921          N2 9P
7984     London NW1
11968       W1T ATT
14579       WIT 6EB
20903      Southall
29007         E8 QN
29028       8AG, UK
29556       4AA, UK
29727       N16 ONB
29734        E5 OQJ
30361       E8 2 NP
30822      UNIT 48)
38056     RM4 1 1QH
39192        HA4TAW
39336       UB4 OJT
42694       TW5 0A5
46415       , Manst
73130      SW17 ORH
Name: PostCode, dtype: str


-> based on the 'unclassified' values:

These are largely due to values that are not postcodes at all (`UNIT 48)`, `, Manst`), letter-as-number or number-as-letter confusion (`N16 ONB`, `WIT 6EB`), or slightly broken postcodes (`E8 2 NP`, `RM4 1 1QH`).  

As these represent 18 of 82k values (around 0.02%), they are not worth bespoke handling, unless (for some reason) they were of particular interest.  

Lastly, in terms of a 'blind spot' where both `geocode` and `PostCode` are missing (or part missing):

In [12]:
# re-generate missing geocode df (now with postcode tier included)
geo_missing = df[df["geocode.longitude"].isna()]

# print counts by postcode tier
print(geo_missing["postcode_tier"].value_counts())

postcode_tier
outward         8469
full            3957
sector          1113
blank            519
unclassified      11
Name: count, dtype: int64


-> based on the postcode tier counts for missing geocode rows:

The location gap is mostly recoverable, but at varying precision: 
- 28.1% have a full postcode (straightforward lookup)
- 68.1% have only an outward or sector-level postcode (recoverable at reduced, district/sector precision only)
- 3.8% (blank or unclassified) are unrecoverable 

Notably, 100% of sector-tier rows are also missing geocode. This may be consistent with the BusinessType finding that peripatetic/non-fixed-premises businesses (mobile caterers, farmers, distributors) drive most of this gap, rather than three unrelated data issues.  

As with missing compound scores values, I won't drop rows with partial/missing geocode value at this point in time, but will bear the missing data in mind when it comes to putting together a model down the line.  

### Final Clean Steps

Having looked into the missing compound score / geocode values, I can now proceed with the final clean steps.

First, the `Distance` column can be dropped:

In [13]:
df = df.drop(columns=["Distance"])

Second, the cleaned DataFrame can be saved, this time as a CSV (no longer nested so JSON not needed):

In [14]:
df.to_csv("../data/processed/fsa_london_establishments_clean.csv", index=False)
print(f"Saved {len(df)} rows, {len(df.columns)} columns.")

Saved 81217 rows, 28 columns.


-> final output:

Raw data pull normalised, and saved with `postcode_tier` added and `Distance` dropped.

`scores`/`geocode` nulls remain unresolved by choice, with decisions documented in the notebook above, to be actioned in the modelling and/or ONS location data join later.